# Day 2 — EDA + 결합 데이터 (분석 결과 노트북)

이 노트북은 `prompts/eda_question_bank.md`의 50개 질문을 **코드로 풀어둔 답안 버전**입니다.
- 본인이 먼저 `b2g_day2_eda_questions.ipynb` 로 풀어본 다음, 막히는 곳만 여기서 확인하는 걸 권장
- 결과 변수·임계값 바꿔보기

## 사전 준비
좌측 폴더 아이콘 → 다음 CSV 업로드 (강의 레포 `data/` 폴더에서 받기 가능):
- `기간별_일평균_대기환경_정보_2024년.csv`, `_2025년.csv` (CP949)
- `seoul_bike_daily_2024.csv` (CP949)
- `seoul_subway_daily_total_2024.csv` (CP949)
- `OBS_ASOS_DD_20260510160758.csv` (CP949)
- `time_series_KR_20240101-0000_20260510-1610.csv` (UTF-8)


In [ ]:
# 한글 폰트 (Colab)
!apt -qq install fonts-nanum > /dev/null
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

import pandas as pd, numpy as np
print('OK')

## 데이터 로드 (한 번에)

In [ ]:
# 미세먼지 (2024 + 2025)
a = pd.read_csv('기간별_일평균_대기환경_정보_2024년.csv', encoding='cp949')
b = pd.read_csv('기간별_일평균_대기환경_정보_2025년.csv', encoding='cp949')
a = a.rename(columns={'측정일시':'date','측정소명':'district','미세먼지(㎍/㎥)':'pm10','초미세먼지(㎍/㎥)':'pm25'})[['date','district','pm10','pm25']]
b = b.rename(columns={'측정일시':'date','측정소명':'district','미세먼지농도(㎍/㎥)':'pm10','초미세먼지농도(㎍/㎥)':'pm25'})[['date','district','pm10','pm25']]
air = pd.concat([a, b], ignore_index=True)
air['date'] = pd.to_datetime(air['date'].astype(str), format='%Y%m%d')
air['year'] = air['date'].dt.year; air['month'] = air['date'].dt.month
air['weekday'] = air['date'].dt.weekday
air['season'] = air['month'].map(lambda m: '황사(3-5월)' if 3<=m<=5 else '비황사')

# 따릉이 / 지하철 (2024년)
bike = pd.read_csv('seoul_bike_daily_2024.csv', encoding='cp949').rename(columns={'대여일자':'date','대여건수':'rides'})
bike['date'] = pd.to_datetime(bike['date'])
sub = pd.read_csv('seoul_subway_daily_total_2024.csv', encoding='cp949').rename(columns={'수송일자':'date','총승하차인원':'passengers'})[['date','passengers']]
sub['date'] = pd.to_datetime(sub['date'])

# 날씨 (ASOS 2024+2025)
weather = pd.read_csv('OBS_ASOS_DD_20260510160758.csv', encoding='cp949').rename(columns={
    '일시':'date','평균기온(°C)':'temp_avg','최저기온(°C)':'temp_min','최고기온(°C)':'temp_max',
    '일강수량(mm)':'rain','평균 풍속(m/s)':'wind_avg','최다풍향(16방위)':'wind_dir','평균 상대습도(%)':'humidity_avg',
})[['date','temp_avg','temp_min','temp_max','rain','wind_avg','wind_dir','humidity_avg']]
weather['date'] = pd.to_datetime(weather['date'])

# 검색량 (구글 트렌드 월별)
buzz = pd.read_csv('time_series_KR_20240101-0000_20260510-1610.csv')
buzz.columns = ['date','buzz']; buzz['date'] = pd.to_datetime(buzz['date'])

# 일평균 PM2.5
daily_pm = air.groupby('date')['pm25'].mean().reset_index()
daily_pm['grade'] = pd.cut(daily_pm['pm25'], bins=[-1,15,35,75,1e9], labels=['좋음','보통','나쁨','매우나쁨'])

print('air:', air.shape, '· bike:', bike.shape, '· sub:', sub.shape, '· weather:', weather.shape, '· buzz:', buzz.shape)

---

# A. 미세먼지 단독 (10개)


**A1. 두 해 통합 shape — 행 수 / 컬럼 수 / 측정소 수 / 날짜 범위**

In [ ]:
print('통합 shape:', air.shape)
print('컬럼:', list(air.columns))
print('측정소:', air['district'].nunique())
print('기간:', air['date'].min().date(), '~', air['date'].max().date())

**A2. 컬럼별 결측 비율 / 50% 이상 결측 측정소가 있나?**

In [ ]:
print('전체 결측 비율(%):')
print((air[['pm10','pm25']].isna().mean()*100).round(2))
miss_per_station = air.groupby('district')['pm25'].apply(lambda s: s.isna().mean()*100)
print('\n측정소별 PM2.5 결측률 Top 5(%):'); print(miss_per_station.sort_values(ascending=False).head())
print('\n50% 이상 결측 측정소:', (miss_per_station >= 50).sum())

**A3. PM2.5·PM10 분포 (히스토그램 + skew)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, label in zip(axes, ['pm25','pm10'], ['PM2.5','PM10']):
    ax.hist(air[col].dropna(), bins=60, color='#4A7BF7', alpha=0.7)
    ax.set_title(f'{label} 분포 (skew={air[col].skew():.2f})')
    ax.set_xlabel(f'{label} (㎍/㎥)'); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

**A4. 자치구별 PM2.5 연평균 Top 5 / Bottom 5 / 격차**

In [ ]:
g = air.groupby('district')['pm25'].mean().sort_values()
print('Bottom 5 (좋은 자치구):'); print(g.head())
print('\nTop 5 (나쁜 자치구):'); print(g.tail())
print(f'\n격차: {(g.max()-g.min())/g.mean()*100:.1f}% (최악-최선)/평균')

**A5. 2024 → 2025 자치구별 PM2.5 변화율 — 가장 좋아진 곳·나빠진 곳**

In [ ]:
yoy = air.groupby(['district','year'])['pm25'].mean().unstack('year')
yoy['change_pct'] = (yoy[2025]-yoy[2024])/yoy[2024]*100
yoy_sorted = yoy.sort_values('change_pct')
print('가장 좋아진 자치구:'); print(yoy_sorted.head(3)[[2024, 2025, 'change_pct']].round(2))
print('\n가장 나빠진 자치구:'); print(yoy_sorted.tail(3)[[2024, 2025, 'change_pct']].round(2))

**A6. 월별 PM2.5 평균/표준편차 — 시즌성 패턴**

In [ ]:
monthly = air.groupby('month')['pm25'].agg(['mean','std']).round(2)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly.index, monthly['mean'], yerr=monthly['std'], color='#4A7BF7', alpha=0.7, capsize=4)
ax.axvspan(2.5, 5.5, color='#FBBF24', alpha=0.15, label='황사 시즌')
ax.set_xticks(range(1,13)); ax.set_xticklabels([f'{m}월' for m in range(1,13)])
ax.set_ylabel('PM2.5 월평균 ± 표준편차')
ax.set_title('월별 PM2.5 시즌성'); ax.legend()
plt.tight_layout(); plt.show()
print(monthly)

**A7. 황사 시즌 vs 비황사 시즌 PM10 분포 (박스플롯)**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data = [air[air['season']=='황사(3-5월)']['pm10'].dropna(),
        air[air['season']=='비황사']['pm10'].dropna()]
ax.boxplot(data, tick_labels=['황사 시즌','비황사 시즌'], showfliers=False)
ax.set_ylabel('PM10 (㎍/㎥)'); ax.set_title('PM10 시즌별 분포')
ax.grid(axis='y', alpha=0.25, linestyle='--')
plt.tight_layout(); plt.show()

**A8. 요일별 PM2.5 평균 — 평일 vs 주말 t-test**

In [ ]:
from scipy import stats
weekday_avg = air.groupby('weekday')['pm25'].mean()
weekday_avg.index = ['월','화','수','목','금','토','일']
print('요일별 PM2.5 평균:'); print(weekday_avg.round(2))

weekday_vals = air[air['weekday']<5]['pm25'].dropna()
weekend_vals = air[air['weekday']>=5]['pm25'].dropna()
t, p = stats.ttest_ind(weekday_vals, weekend_vals)
print(f'\n평일 평균: {weekday_vals.mean():.2f}  /  주말 평균: {weekend_vals.mean():.2f}')
print(f't-statistic: {t:.3f}  p-value: {p:.4f}  → {"유의함" if p<0.05 else "유의하지 않음"}')

**A9. PM10·PM2.5 상관계수 + 산점도**

In [ ]:
corr = air[['pm10','pm25']].corr().iloc[0,1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(air['pm10'], air['pm25'], alpha=0.2, s=8, color='#4A7BF7')
ax.set_xlabel('PM10'); ax.set_ylabel('PM2.5')
ax.set_title(f'PM10 vs PM2.5  (corr={corr:.3f})')
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

**A10. 일별 spike Top 10 — 진짜 이상치인가, 황사인가?**

In [ ]:
daily_max = air.groupby('date')['pm25'].max().sort_values(ascending=False).head(10)
print('PM2.5 일별 spike Top 10:')
for d, v in daily_max.items():
    is_dust = '황사' if 3<=d.month<=5 else '비황사'
    print(f'  {d.date()} ({is_dust}): PM2.5 = {v:.0f}')

---

# B. 미세먼지 + 따릉이 (5개) — 2024년 매칭


**B11. PM2.5 등급별 따릉이 일평균 대여 건수**

In [ ]:
bike_d = bike.merge(daily_pm[daily_pm['date'].dt.year==2024][['date','grade']], on='date')
bike_g = bike_d.groupby('grade', observed=True)['rides'].mean().reindex(['좋음','보통','나쁨','매우나쁨'])
print(bike_g.round(0))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(bike_g.index, bike_g.values, color=['#10B981','#4A7BF7','#FFB020','#FF6B6B'])
ax.set_ylabel('일평균 대여 건수'); ax.set_title('PM2.5 등급별 따릉이 (2024)')
plt.tight_layout(); plt.show()

**B12. 미세먼지 등급 1단계 악화 시 따릉이 대여가 몇 % 감소?**

In [ ]:
base = bike_g.iloc[0]
for grade, v in bike_g.items():
    delta = (v - base) / base * 100
    print(f'  {grade}: {v:,.0f} ({delta:+.1f}%)')

**B13. 황사 vs 비황사 시즌 따릉이 이용 + t-test**

In [ ]:
bike_season = bike.merge(air[['date','season']].drop_duplicates(), on='date')
fig, ax = plt.subplots(figsize=(7, 4))
data = [bike_season[bike_season['season']=='황사(3-5월)']['rides'],
        bike_season[bike_season['season']=='비황사']['rides']]
ax.boxplot(data, tick_labels=['황사 시즌','비황사 시즌'], showfliers=False)
ax.set_ylabel('일별 대여 건수'); ax.set_title('따릉이 시즌별 분포 (2024)')
plt.tight_layout(); plt.show()
t, p = stats.ttest_ind(data[0], data[1])
print(f't={t:.3f}, p={p:.4f}')

**B14. 미세먼지 영향이 더 큰 시즌은? (계절별)**

In [ ]:
bike_q = bike.merge(daily_pm[['date','grade']], on='date')
bike_q['quarter'] = bike_q['date'].dt.quarter.map({1:'Q1(겨울)',2:'Q2(봄)',3:'Q3(여름)',4:'Q4(가을)'})
pivot = bike_q.groupby(['quarter','grade'], observed=True)['rides'].mean().unstack('grade')
print(pivot.round(0))

**B15. "PM2.5 30 이하 + 맑음" 조합에서 대여가 폭발하나?** (조건 결합)

In [ ]:
# 날씨 결합 — 비강수 + 풍속 보통 + 좋은 PM
bike_w = bike.merge(weather[['date','rain','wind_avg']], on='date').merge(daily_pm[['date','pm25']], on='date')
clear_low_pm = bike_w[(bike_w['pm25']<=30) & (bike_w['rain']<1)]['rides'].mean()
overall = bike_w['rides'].mean()
print(f'전체 평균: {overall:,.0f}')
print(f'PM2.5≤30 + 비없음: {clear_low_pm:,.0f}  ({(clear_low_pm/overall-1)*100:+.1f}%)')

---

# C. 미세먼지 + 지하철 (5개) — 2024년 매칭


**C16. PM2.5 등급별 지하철 일별 총 승하차 — 회피 효과 있나?**

In [ ]:
sub_d = sub.merge(daily_pm[daily_pm['date'].dt.year==2024][['date','grade']], on='date')
sub_g = sub_d.groupby('grade', observed=True)['passengers'].mean().reindex(['좋음','보통','나쁨','매우나쁨'])
print((sub_g/1e6).round(2), '(M명)')
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sub_g.index, sub_g.values, color=['#10B981','#4A7BF7','#FFB020','#FF6B6B'])
ax.set_ylabel('일평균 총 승하차'); ax.set_title('PM2.5 등급별 지하철 (2024)')
plt.tight_layout(); plt.show()

**C17. 따릉이=민감 / 지하철=둔감 — 대체재 가설 검증**

In [ ]:
base_b, base_s = bike_g.iloc[0], sub_g.iloc[0]
print('등급별 변화율 (좋음 대비)')
for grade in bike_g.index:
    db = (bike_g[grade] - base_b) / base_b * 100
    ds = (sub_g[grade] - base_s) / base_s * 100
    print(f'  {grade}: 따릉이 {db:+.1f}%  /  지하철 {ds:+.1f}%')

**C18. 황사 spike 일자에 따릉이↓ + 지하철↑ 동시 일어나나?**

In [ ]:
spike_dates = daily_pm[daily_pm['date'].dt.year==2024].nlargest(10, 'pm25')[['date','pm25']]
spike_join = spike_dates.merge(bike, on='date').merge(sub, on='date')
print(spike_join.round(0).to_string(index=False))

**C19. 출근시간 7~9시 PM2.5와 지하철 승차 관계** ⛔

**답 안 됨**: AQ 데이터는 일별이라 시간대 분석 불가. 시간대 분석을 원하면 에어코리아 시간별 OpenAPI를 별도로 받아야 함.

**C20. 평일 vs 주말 — 두 데이터 다 평일이 더 높나? (커뮤팅)**

In [ ]:
bike_wk = bike.copy(); bike_wk['is_weekday'] = bike_wk['date'].dt.weekday<5
sub_wk = sub.copy(); sub_wk['is_weekday'] = sub_wk['date'].dt.weekday<5
print('따릉이 평일/주말 평균:')
print(bike_wk.groupby('is_weekday')['rides'].mean().rename({True:'평일', False:'주말'}).round(0))
print('\n지하철 평일/주말 평균:')
print((sub_wk.groupby('is_weekday')['passengers'].mean()/1e6).rename({True:'평일', False:'주말'}).round(2), '(M)')

---

# D. 미세먼지 + 날씨 (8개) — 2024 + 2025


**D21. 풍속 → PM2.5 음의 상관 / 바람이 셀수록 농도 떨어지나?**

In [ ]:
j = daily_pm.merge(weather, on='date')
print(f'풍속-PM2.5 상관: {j["wind_avg"].corr(j["pm25"]):+.3f}')
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(j['wind_avg'], j['pm25'], alpha=0.4, s=15, color='#4A7BF7')
ax.set_xlabel('평균풍속 (m/s)'); ax.set_ylabel('PM2.5')
ax.set_title('풍속 vs PM2.5')
plt.tight_layout(); plt.show()

**D22. 강수일 vs 비강수일 PM2.5 분포 (비가 씻어내나?)**

In [ ]:
rain_day = j[j['rain'] >= 1]['pm25'].dropna()
no_rain = j[j['rain'] < 1]['pm25'].dropna()
print(f'강수일 평균: {rain_day.mean():.2f}  ({len(rain_day)}일)')
print(f'비강수일 평균: {no_rain.mean():.2f}  ({len(no_rain)}일)')
t, p = stats.ttest_ind(rain_day, no_rain)
print(f't={t:.3f}, p={p:.4f}')

**D23. 풍향(서·북서 vs 동) 별 PM10 — 황사 유입 검증**

In [ ]:
# 풍향 16방위: 서~북서 = 247.5~315 (즉 250~315 근사)
j['wind_group'] = j['wind_dir'].apply(lambda d: '서·북서(황사 유입)' if 220<=d<=340 else '그 외')
print(j.merge(air[['date','pm10']].groupby('date').mean().reset_index(), on='date')
      .groupby('wind_group')['pm10'].mean().round(2))

**D24. 기온·습도와 PM 상관 — 겨울 난방 효과?**

In [ ]:
print('PM2.5 ↔ 날씨 변수 상관계수:')
for c, label in [('temp_avg','평균기온'), ('humidity_avg','평균습도'), ('rain','강수')]:
    print(f'  {label}: {j[c].corr(j["pm25"]):+.3f}')

**D25. "맑고 풍속 약함 + 한파" 조합에서 PM 폭증?** (정체)

In [ ]:
cold_calm = j[(j['temp_avg']<-5) & (j['wind_avg']<2) & (j['rain']<1)]
overall = j['pm25'].mean()
print(f'전체 PM2.5 평균: {overall:.2f}')
print(f'한파+약풍+맑음({len(cold_calm)}일) PM2.5: {cold_calm["pm25"].mean():.2f}  (+{(cold_calm["pm25"].mean()/overall-1)*100:.1f}%)')

**D26. 황사 시즌 풍향 분포 — 작년과 다른가?**

In [ ]:
j['year'] = j['date'].dt.year
j['month'] = j['date'].dt.month
dust = j[(j['month']>=3) & (j['month']<=5)]
print(dust.groupby('year')['wind_dir'].agg(['mean','median','std']).round(1))

**D27. 일교차 큰 날 PM 더 높나? (대기 정체)**

In [ ]:
j['temp_range'] = j['temp_max'] - j['temp_min']
j['range_q'] = pd.qcut(j['temp_range'], 4, labels=['Q1(작음)','Q2','Q3','Q4(큼)'])
print(j.groupby('range_q', observed=True)['pm25'].mean().round(2))

**D28. "다음 날 PM 예측" — 오늘 날씨 변수로 회귀 R²**

In [ ]:
from sklearn.linear_model import LinearRegression
j_sorted = j.dropna(subset=['temp_avg','wind_avg','rain','humidity_avg','pm25']).sort_values('date').reset_index(drop=True)
j_sorted['pm25_next'] = j_sorted['pm25'].shift(-1)
j_sorted = j_sorted.dropna(subset=['pm25_next'])
X = j_sorted[['temp_avg','wind_avg','rain','humidity_avg']]
y = j_sorted['pm25_next']
m = LinearRegression().fit(X, y)
print(f'R² = {m.score(X, y):.3f}')
print('회귀계수:', dict(zip(X.columns, m.coef_.round(3))))

---

# E. 미세먼지 + 언급량 (4개) — 월 단위


**E29. "미세먼지" 검색량과 PM2.5 일별 상관 (월 단위로 집계)**

In [ ]:
air_m = air.assign(mp=air['date'].dt.to_period('M')).groupby('mp')['pm25'].mean().reset_index()
buzz_m = buzz.assign(mp=buzz['date'].dt.to_period('M')).groupby('mp')['buzz'].mean().reset_index()
jb = air_m.merge(buzz_m, on='mp').sort_values('mp')
jb['date'] = jb['mp'].dt.to_timestamp()
print(f'월별 PM2.5 vs "미세먼지" 검색량 상관: {jb["pm25"].corr(jb["buzz"]):+.3f}')

**E30. 사람들은 PM 몇 ㎍/㎥부터 검색하기 시작하나? (임계값)**

In [ ]:
# PM2.5 구간별 평균 검색 인덱스
jb['pm_bin'] = pd.cut(jb['pm25'], bins=[0, 15, 25, 35, 50, 100], labels=['~15','15~25','25~35','35~50','50+'])
print(jb.groupby('pm_bin', observed=True)['buzz'].mean().round(1))

**E31. "마스크"·"공기청정기" 검색 급등 vs PM 실측 — 어느 게 며칠 빠른가?**

**별도 키워드 buzz 데이터 필요** — 현재는 "미세먼지" 단일 키워드만 받음. 네이버 데이터랩에서 "마스크", "공기청정기" 키워드 추가로 받아 동일 흐름으로 분석 가능.

**E32. 같은 PM 농도에도 시즌·요일에 따라 검색 강도 다른가?**

In [ ]:
# 월별로만 가능 (요일은 월 집계 후 의미 X). 황사 vs 비황사 시즌별 PM 대비 검색 강도
jb['season'] = jb['date'].dt.month.map(lambda m: '황사(3-5월)' if 3<=m<=5 else '비황사')
jb['buzz_per_pm'] = jb['buzz'] / jb['pm25']
print(jb.groupby('season')['buzz_per_pm'].mean().round(2), '(검색량/PM)')

---

# F. 쇼핑 적재본 (8개) — Day 4

> 쇼핑 적재본(다이소·올리브영·컬리·네이버쇼핑)은 4일차 결합용으로 강의 기간 누적.
> Day 2 시점에는 데이터 미보유. Day 4 노트북에서 다룸.


---

# G. 답 안 되는 질문 (5개) — 한계 인식

| 질문 | 왜 답 못 함 |
|---|---|
| 출근시간(7~9시) vs 퇴근시간 PM | AQ 일별 — 시간대 분석 불가. 에어코리아 시간별 OpenAPI 별도 |
| 코로나 봉쇄 전후(2020~2022) PM 변화 | 배포본은 2024~2025만. 과거 자료 별도 다운로드 |
| 자치구별 사망률·호흡기질환 상관 | 보건 데이터 별도 — 강의 범위 외 |
| 중국 황사 발원지 풍향 | 위성·재분석 데이터 필요 — 강의 범위 외 |
| 학생 본인 회사 매출과 PM 상관 | 사내 데이터 — 미션 보너스 영역 |


---

# H. 워밍업 (5개)


**H46. 내가 사는 자치구의 2025 PM2.5 평균은?** — 본인 자치구로 변경

In [ ]:
my_district = '강남구'  # 본인 자치구로 변경
result = air[(air['district']==my_district) & (air['year']==2025)]['pm25'].mean()
print(f'{my_district} 2025 PM2.5 평균: {result:.2f} ㎍/㎥')

**H47. 어제 vs 지난주 같은 요일 비교** — 짧은 기간 변동 체감 (데이터 마지막 날 기준)

In [ ]:
last = air['date'].max()
last_pm = air[air['date']==last]['pm25'].mean()
last_week = air[air['date']==last - pd.Timedelta(days=7)]['pm25'].mean()
print(f'{last.date()} (마지막일): PM2.5 {last_pm:.2f}')
print(f'{(last - pd.Timedelta(days=7)).date()} (지난주 같은 요일): {last_week:.2f}')

**H48. 가장 깨끗했던 하루 + 그날 어떤 날씨였는지**

In [ ]:
best = daily_pm.nsmallest(1, 'pm25').iloc[0]
print(f'가장 깨끗한 날: {best["date"].date()}  PM2.5={best["pm25"]:.1f}')
w = weather[weather['date']==best['date']]
if not w.empty:
    print('그날 날씨:', w[['temp_avg','rain','wind_avg','humidity_avg']].iloc[0].to_dict())

**H49. 황사 경보 수준이었던 날 따릉이 대여가 정말 줄었나?**

In [ ]:
# 매우나쁨(PM2.5 75+) 일자
bad_dates = daily_pm[(daily_pm['date'].dt.year==2024) & (daily_pm['pm25']>75)][['date','pm25']]
bad_bike = bad_dates.merge(bike, on='date')
print(f'매우나쁨 일수 (2024): {len(bad_bike)}일')
print(f'매우나쁨 일 따릉이 평균: {bad_bike["rides"].mean():,.0f}')
print(f'전체 평균: {bike["rides"].mean():,.0f}')

**H50. 한 달 뒤 PM2.5 예측 — 가진 변수로 어디까지 가능한가?**

**Day 3·4 예고**: 이 노트북의 D28 회귀 모델을 확장해서 한 달 뒤 PM 예측. 가진 변수로 R² ~0.3 정도가 한계 — 더 나아가려면 위성·외기 데이터 결합 필요. Day 4 의사결정 리포트와 연결.

---

## CLAUDE.md 누적

오늘 발견한 규칙을 자연어 한 마디로 CLAUDE.md에 추가:

```
오늘 EDA에서 발견한 규칙을 CLAUDE.md에 추가해줘.
- 측정일시는 항상 datetime 파싱 (format='%Y%m%d')
- 결측 50% 이상 측정소 제외
- 이상치 IQR 3배 (황사 spike 인정), 황사 시즌 = 3~5월
- 결합 데이터 날짜 컬럼은 'date'로 통일
- 미세먼지는 CP949, buzz는 UTF-8
- y축 단위 항상 명시 (PM은 ㎍/㎥, 따릉이는 건, 지하철은 명)
```
